# CLAW + LeWM on Google Colab
Run the cells top to bottom (Runtime > Run all works). **Before starting:** Runtime > Change runtime type > GPU, and make sure the latest code is **pushed to GitHub**: Colab clones the repo and only sees pushed commits.

Steps: Settings, clone, install, PushT data (~13 GB compressed), smoke test, train (checkpoints to Drive, resumable), export the latest checkpoint, CLAW eval (CEM over latent actions, nearest-neighbour map to real actions, video + success rate).

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU - switch the runtime type to GPU first"
!df -h /content | tail -1

## Settings
Everything you may want to change is here. `IMG_SIZE` is used for training **and** evaluation.

In [ ]:
CLAW_RUN = "claw_full"      # run name: weights go to MyDrive/lewm/checkpoints/<CLAW_RUN>/
CLAW_EPOCHS = 5
IMG_SIZE = 126              # train AND eval image size (the paper uses 224; patch size is 14, so 126 = 9 x 14 keeps every pixel)
CLAW_BATCH = 128            # lower (e.g. 64) if you hit an out-of-memory error
TRAIN_BATCHES = None        # None = full epochs; e.g. 100 for a quick dev run
CKPT_EVERY = 500            # save a resumable checkpoint to Drive every N steps and at every epoch end
NUM_REF = 4096              # reference windows for the latent -> real action lookup
EVAL_EPISODES = 50          # PushT eval episodes (the paper uses 50)
LIMITS = "" if TRAIN_BATCHES is None else f"+trainer.limit_train_batches={TRAIN_BATCHES} +trainer.limit_val_batches=20"

# shared training args (bf16 on GPUs that support it; the T4 falls back to fp16 mixed precision)
import torch
PRECISION = "bf16" if torch.cuda.is_bf16_supported() else "16-mixed"
COMMON = (
    "data=pusht data.dataset.name=pusht_expert_train.h5 "
    f"trainer.precision={PRECISION} num_workers=2 wandb.enabled=False"
)
print("precision:", PRECISION)

## 1. Clone the repo
Re-running pulls the latest pushed code instead of failing.

In [ ]:
%cd /content
!test -d le-wm && git -C le-wm pull || git clone https://github.com/arshagarwal/le-wm.git
%cd /content/le-wm
!git log --oneline | head -3

## 2. Install dependencies
This takes a few minutes. `imageio[ffmpeg]` writes the eval videos; `pygame pymunk shapely opencv-python-headless` run the PushT simulator (they are not part of the `train` extra).

In [ ]:
!pip install -q "stable-worldmodel[train]" hdf5plugin h5py hf_transfer "imageio[ffmpeg]" pygame pymunk shapely opencv-python-headless
!apt-get -qq install -y zstd > /dev/null
import stable_worldmodel, stable_pretraining, hdf5plugin
from stable_worldmodel.data.format import list_formats
print("formats:", list_formats())
assert "hdf5" in list_formats(), "hdf5 format not registered - check hdf5plugin/h5py install"
print("deps OK")

## 3. Download and decompress the PushT dataset
The dataset is an `.h5.zst` archive (~13 GB). It goes under `$STABLEWM_HOME/datasets/`, which is where this version of stable-worldmodel looks.

In [ ]:
import os
os.environ["STABLEWM_HOME"] = "/content/stablewm"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
DATA_DIR = "/content/stablewm/datasets"
os.makedirs(DATA_DIR, exist_ok=True)

!hf download quentinll/lewm-pusht --repo-type dataset --local-dir /content/dl_pusht
!ls -lh /content/dl_pusht

In [ ]:
# decompress; handles either a plain zstd'd .h5 or a zstd'd tar
!zstd -d -f /content/dl_pusht/pusht_expert_train.h5.zst -o /content/dl_pusht/out.bin
with open("/content/dl_pusht/out.bin", "rb") as f:
    magic = f.read(8)
if magic.startswith(b"\x89HDF"):
    !mv /content/dl_pusht/out.bin {DATA_DIR}/pusht_expert_train.h5
else:
    !tar -xf /content/dl_pusht/out.bin -C {DATA_DIR} && rm /content/dl_pusht/out.bin
!rm -f /content/dl_pusht/pusht_expert_train.h5.zst
!ls -lh {DATA_DIR}
!df -h /content | tail -1

In [ ]:
# sanity check: dataset loads and shows its columns
import stable_worldmodel as swm
ds = swm.data.load_dataset("pusht_expert_train.h5", frameskip=5, num_steps=4,
                           keys_to_load=["pixels", "action", "proprio", "state"],
                           keys_to_cache=["action", "proprio", "state"])
print(len(ds), "samples")
print({k: tuple(getattr(v, "shape", ())) for k, v in ds[0].items()})

## 3. Smoke test (about 1 minute)
Runs 5 train batches and 2 val batches to confirm training works before the full run.

In [ ]:
!cd /content/le-wm && STABLEWM_HOME=/content/stablewm python train.py {COMMON} img_size={IMG_SIZE} \
    trainer.max_epochs=2 +trainer.limit_train_batches=5 +trainer.limit_val_batches=2 \
    output_model_name=smoke subdir=smoke

## 4. Train
Checkpoints and the run state go to Google Drive. Training saves a resumable checkpoint every `CKPT_EVERY` steps and at each epoch end, so after a disconnect you re-run: Settings, clone, install, data download, this Drive cell, and the training cell. Training continues from the last save with a fresh data shuffle (`subdir` is the same).
The weights for evaluation are exported from that checkpoint in the next section, so you can evaluate before an epoch has finished.

In [ ]:
from google.colab import drive
import os
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/lewm", exist_ok=True)
os.environ["STABLEWM_HOME"] = "/content/drive/MyDrive/lewm"
os.environ["SPT_CACHE_DIR"] = "/content/spt_cache"   # Lightning's own per-epoch files stay local; the resumable one goes to Drive
# dataset stays on local disk (fast); symlink it into the Drive-backed home
os.makedirs("/content/drive/MyDrive/lewm/datasets", exist_ok=True)
!ln -sf /content/stablewm/datasets/pusht_expert_train.h5 /content/drive/MyDrive/lewm/datasets/pusht_expert_train.h5
!ls -l /content/drive/MyDrive/lewm/datasets

In [ ]:
ckpt = f"/content/drive/MyDrive/lewm/checkpoints/{CLAW_RUN}/{CLAW_RUN}_weights.ckpt"
print("resuming from", ckpt if os.path.exists(ckpt) else "nothing: fresh run")
!cd /content/le-wm && STABLEWM_HOME=/content/drive/MyDrive/lewm python train.py {COMMON} img_size={IMG_SIZE} \
    loader.batch_size={CLAW_BATCH} trainer.max_epochs={CLAW_EPOCHS} {LIMITS} ckpt_every_n_steps={CKPT_EVERY} \
    output_model_name={CLAW_RUN} subdir={CLAW_RUN}

## 5. Evaluate (CLAW-style)
1. **Export** the latest saved training state to `weights_latest.pt`, `lam_latest.pt` and `config.json` (what `eval_claw.py` loads). This works mid-epoch too.
2. **Eval** (`eval_claw.py`): builds the latent-to-real-action lookup from reference windows, then per episode plans with CEM over latent actions, maps them to real actions by nearest neighbour, runs the first actions in the PushT simulator, replans from the new frame (MPC), and saves a video. Results and videos: `MyDrive/lewm/<CLAW_RUN>/`.

In [ ]:
import sys, json
sys.path.insert(0, "/content/le-wm")
from omegaconf import OmegaConf
from jepa import LAM
d = f"/content/drive/MyDrive/lewm/checkpoints/{CLAW_RUN}"
ck = torch.load(f"{d}/{CLAW_RUN}_weights.ckpt", map_location="cpu", weights_only=False)
cfg = OmegaConf.load(f"{d}/config.yaml")
assert cfg.img_size == IMG_SIZE, f"IMG_SIZE={IMG_SIZE} in Settings but this run was trained at {cfg.img_size}"
sd = ck["state_dict"]
torch.save({k[len("model."):]: v for k, v in sd.items() if k.startswith("model.")}, f"{d}/weights_latest.pt")
json.dump(OmegaConf.to_container(cfg.model, resolve=True), open(f"{d}/config.json", "w"), indent=2)
lam = LAM(emb_dim=cfg.embed_dim, latent_dim=cfg.latent_dim, cond_dim=cfg.embed_dim)
lam.load_state_dict({k[len("lam."):]: v for k, v in sd.items() if k.startswith("lam.")})
torch.save({"config": lam.config, "state_dict": lam.state_dict()}, f"{d}/lam_latest.pt")
print(f"exported {CLAW_RUN}: global_step {ck['global_step']}, epoch {ck['epoch']}, img_size {cfg.img_size}")

In [ ]:
!cd /content/le-wm && STABLEWM_HOME=/content/drive/MyDrive/lewm SDL_VIDEODRIVER=dummy python eval_claw.py --config-name=pusht \
    policy={CLAW_RUN}/weights_latest.pt eval.img_size={IMG_SIZE} +num_ref={NUM_REF} \
    eval.num_eval={EVAL_EPISODES} output.filename=pusht_claw_results.txt

## 6. Check outputs
Weights and LAM files: `MyDrive/lewm/checkpoints/<CLAW_RUN>/`. Results and one video per episode: `MyDrive/lewm/<CLAW_RUN>/`.

In [ ]:
!ls -lh /content/drive/MyDrive/lewm/checkpoints/{CLAW_RUN}
!ls -lh /content/drive/MyDrive/lewm/{CLAW_RUN}
!tail -n 3 /content/drive/MyDrive/lewm/{CLAW_RUN}/pusht_claw_results.txt

## Optional: weights & biases logging
Add `wandb.enabled=True wandb.config.entity=<your_entity> wandb.config.project=<your_project>` to the training command and run `!wandb login` first.